### Problem Statement

How to normalize the scores of sparse embedding

### Objective

comparing dense VS Sparse BM42 retrival methods and analyzing the scores

#### What is done

* Created a synthetic dataset
* Generated sparse and dense embeddings
* stored in Qdrant
* Implemented dense and sparse retrieval
* Compared results

#### next to be done

* Score normalization

In [59]:
# pip install qdrant-client pandas numpy scikit-learn fastembed

In [60]:
from fastembed import SparseTextEmbedding, TextEmbedding
from qdrant_client import QdrantClient,models
import numpy as np
import pandas as pd

In [61]:
# pip install fastembed

In [62]:
# Testing the TextEmbedding model

from fastembed import TextEmbedding

model = TextEmbedding()
embeddings = list(model.embed(["hello world"]))
print(len(embeddings[0]))

384


In [63]:
documents = [
{"id": 1, "text": "Diabetes is a chronic condition characterized by high blood sugar levels."},
{"id": 2, "text": "Symptoms of diabetes include frequent urination, thirst, and fatigue."},
{"id": 3, "text": "Heart disease is caused by blocked blood vessels and can lead to heart attack."},
{"id": 4, "text": "Hypertension refers to high blood pressure and increases risk of stroke."},
{"id": 5, "text": "Stroke occurs when blood flow to the brain is interrupted."},
{"id": 6, "text": "Cancer involves uncontrolled growth of abnormal cells in the body."},
{"id": 7, "text": "Breast cancer is one of the most common cancers among women."},
{"id": 8, "text": "Lung cancer is often caused by smoking and air pollution."},
{"id": 9, "text": "Obesity is linked to diabetes, heart disease, and metabolic disorders."},
{"id": 10, "text": "Insulin helps regulate blood sugar levels in the body."},
{"id": 11, "text": "Asthma is a respiratory condition causing difficulty in breathing."},
{"id": 12, "text": "COVID-19 is a viral infection affecting the respiratory system."},
{"id": 13, "text": "Symptoms of COVID-19 include fever, cough, and loss of smell."},
{"id": 14, "text": "Vaccination helps prevent infectious diseases."},
{"id": 15, "text": "Antibiotics are used to treat bacterial infections."},
{"id": 16, "text": "Mental health includes conditions like depression and anxiety."},
{"id": 17, "text": "Depression causes persistent sadness and lack of interest."},
{"id": 18, "text": "Anxiety disorder leads to excessive worry and nervousness."},
{"id": 19, "text": "Kidney disease affects the ability to filter waste from blood."},
{"id": 20, "text": "Liver disease can result from alcohol abuse or infections."},
{"id": 21, "text": "Cholesterol buildup can lead to cardiovascular diseases."},
{"id": 22, "text": "Exercise improves cardiovascular health and reduces disease risk."},
{"id": 23, "text": "A balanced diet helps maintain overall health and prevent illness."},
{"id": 24, "text": "Insomnia is a sleep disorder that affects daily functioning."},
{"id": 25, "text": "Migraine is a neurological condition causing severe headaches."}
]


queries = [
"diabetes symptoms",
"what causes heart disease",
"signs of cancer",
"high blood pressure meaning",
"stroke causes",
"covid symptoms",
"mental health issues",
"what is insulin",
"breathing problems disease",
"treatment for infections",
"diabtes symtoms", # typo test
"ways to prevent disease"
]


In [64]:
model_bm42 = SparseTextEmbedding(model_name="Qdrant/bm42-all-minilm-l6-v2-attentions")
model_dense = TextEmbedding(model_name="jinaai/jina-embeddings-v2-base-en")

In [65]:
# Creating embeddings for documents using both models
dense_vectors = [list(model_dense.embed(doc["text"]))[0] for doc in documents]
sparse_vectors = [list(model_bm42.embed(doc["text"]))[0] for doc in documents]

In [66]:
# Setting up Qdrant client and collection
client = QdrantClient(":memory:")


In [67]:
from qdrant_client.models import Distance, VectorParams,SparseVectorParams

client.create_collection(
    collection_name = "healthcare_docs",
    vectors_config = {"dense": VectorParams(
        size=len(dense_vectors[0]),
        distance=Distance.COSINE
    )},
    sparse_vectors_config={"bm42": SparseVectorParams()})

True

In [68]:
from qdrant_client.models import PointStruct
points = []
for i, doc in enumerate(documents):
    points.append(
    PointStruct(
        id=i,
        vector={"dense": dense_vectors[i],
        "bm42": sparse_vectors[i].as_object()},
        payload={"text": doc["text"]}
    ))
client.upsert(collection_name="healthcare_docs", points=points)

UpdateResult(operation_id=0, status=<UpdateStatus.COMPLETED: 'completed'>)

query = "Diabetes symptoms"

dense_query = list(model_dense.embed([query]))[0]
sparse_query = list(model_bm42.embed([query]))[0]

dense_results = client.query_points(collection_name="healthcare_docs",
                                    query = dense_query,
                                    using ="dense",
                                      limit=5 )

from qdrant_client.models import SparseVector
sparse_vectors = SparseVector(indices=sparse_query.indices,
                                   values=sparse_query.values)
sparse_results = client.query_points(collection_name="healthcare_docs",
                                    query = sparse_vectors,
                                    using = "bm42",
                                    limit=5 )

In [69]:
def run_query(query):
    dense_query = list(model_dense.embed([query]))[0]
    sparse_query = list(model_bm42.embed([query]))[0]

    dense_results = client.query_points(collection_name="healthcare_docs",
                                        query = dense_query,
                                        using ="dense",
                                          limit=5 )

    sparse_vectors = SparseVector(indices=sparse_query.indices,
                                   values=sparse_query.values)
    sparse_results = client.query_points(collection_name="healthcare_docs",
                                        query = sparse_vectors,
                                        using = "bm42",
                                        limit=5 )
    
    print(f"Results for query: '{query}'")
    print("Dense Search Results:")
    for res in dense_results.points:
        print(f"ID: {res.id}, Score: {res.score}, Text: {res.payload['text']}")
    
    print("\nSparse Search Results:")
    for res in sparse_results.points:
        print(f"ID: {res.id}, Score: {res.score}, Text: {res.payload['text']}")

In [70]:
for q in queries:
    run_query(q)
    

Results for query: 'diabetes symptoms'
Dense Search Results:
ID: 1, Score: 0.9054754491772109, Text: Symptoms of diabetes include frequent urination, thirst, and fatigue.
ID: 0, Score: 0.8038575539634764, Text: Diabetes is a chronic condition characterized by high blood sugar levels.
ID: 8, Score: 0.730502054334307, Text: Obesity is linked to diabetes, heart disease, and metabolic disorders.
ID: 9, Score: 0.729816029375196, Text: Insulin helps regulate blood sugar levels in the body.
ID: 12, Score: 0.6861968970323193, Text: Symptoms of COVID-19 include fever, cough, and loss of smell.

Sparse Search Results:
ID: 1, Score: 0.286403626203537, Text: Symptoms of diabetes include frequent urination, thirst, and fatigue.
ID: 0, Score: 0.20173980295658112, Text: Diabetes is a chronic condition characterized by high blood sugar levels.
ID: 8, Score: 0.13276688754558563, Text: Obesity is linked to diabetes, heart disease, and metabolic disorders.
ID: 12, Score: 0.06517910957336426, Text: Sympto